In [11]:
import numpy as np
!pip install xlrd

!pip install pandas
import pandas as pd
import re
from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn import metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from keras import models, Input
from keras import optimizers as opt
from keras import backend as K
from keras.layers import Dense
from keras_tuner.tuners import RandomSearch
from tensorflow.keras.utils import to_categorical

In [12]:
from features import time_series_features, fractal_features, entropy_features, hjorth_features, freq_band_features
import variables as v

# Variables

In [13]:
data_type = "raw"

# Load Dataset

In [15]:
import os
import numpy as np
import scipy.io
def parse_file(fileName):
    fname = fileName.replace(".mat", "")  # remove extension

    pattern = r"([A-Za-z_]+)_sub_(\d+)_trial(\d+)"

    match = re.match(pattern, fname)
    if not match:
        raise ValueError(f"Filename format unrecognized: {fname}")

    task = match.group(1)                # Relax, Maths, Symmetry, Stroop
    subject = int(match.group(2))        # integer subject ID
    trial = int(match.group(3))          # integer trial number

    return task, subject, trial

def load_scales():
    df = pd.read_excel(v.LABELS_PATH)

    # Drop the SECOND row (index 1) which contains "Maths Symmetry Stroop" headers
    # df = df.drop(index=1)
    # Remove any completely empty rows
    df = df.dropna(how='all')

    # Reset indexing
    df = df.reset_index(drop=True)

    # Convert all numeric values

    return df


def load_dataset(data_type="raw"):
    """
    Loads all 120 recordings from the SAM-40 dataset.
    Returns:
        dataset: ndarray (N, 32, T)
        labels: ndarray (N,)
    """

    if data_type == "raw":
        dir = v.DIR_RAW
        data_key = 'Data'
    elif data_type == "wt_filtered":
        dir = v.DIR_FILTERED
        data_key = 'Clean_data'
    else:
        dir = v.DIR_ICA_FILTERED
        data_key = 'Clean_data'

    files = sorted(os.listdir(dir))  # keep ordering stable
    dataset = []
    labels = []
    df = load_scales()

    for f in files:
        if not f.endswith(".mat"):
            continue

        fullpath = os.path.join(dir, f)
        mat = scipy.io.loadmat(fullpath)
        data = mat[data_key]     # shape: (32, T)
        
        dataset.append(data)

        # ---- Assign labels based on filename ----
        if "Relax" in f or "_relaxed" in f.lower():
            labels.append(0)  
        else:
            # Arithmetic OR Memory oR stroop if 
            task, subject, trial = parse_file(f)
            task_map = {
                "Arithmetic": "Maths",
                "Mirror_image": "Symmetry",
                "Stroop": "Stroop"
            }

            task = task_map[task]
            if task == "Symmetry" or task == "Stroop":
                continue


            # if Relax was not triggered, then the task must be Maths/Symmetry/Stroop
            task_idx = {
                "Maths": 0,
                "Symmetry": 1,
                "Stroop": 2
            }[task]

            # Excel columns start at index 1 because column A is "Subject No"
            trial_start_col = {1: 1, 2: 4, 3: 7}[trial]

            # get the correct Excel column
            col = trial_start_col + task_idx

            # df is zero-indexed, subject numbers start at 1
            score = df.iloc[subject, col]
            label = 1 if int(score) > 5 else 0
            labels.append(label)
        
    dataset = np.stack(dataset, axis=0)
    labels = np.array(labels)

    return dataset, labels

dataset_, labels = load_dataset(data_type=data_type)


def split_data(data, sfreq):
    '''
    Splits EEG data into epochs with length 1 sec.

    Args:
        data (ndarray): EEG data.
        sfreq (int): The sampling frequency.
    
    Returns:
        ndarray: The epoched data.

    '''

    n_trials, n_channels, n_samples = data.shape
    print(n_trials, n_channels, n_samples)

    epoched_data = np.empty((n_trials, n_samples//sfreq, n_channels, sfreq))
    for i in range(data.shape[0]):
        for j in range(data.shape[2]//sfreq):
            epoched_data[i, j] = data[i, :, j*sfreq:(j+1)*sfreq]
    print("Epoched data shape:", epoched_data.shape)
    return epoched_data

dataset = split_data(dataset_, v.SFREQ)
print(len(dataset))

ValueError: Worksheet index 0 is invalid, 0 worksheets found

In [156]:
print(len(dataset), len(labels))

480 480


# Compute Features

In [1]:
# features = time_series_features(dataset)
# freq_bands = np.array([1, 4, 8, 12, 30, 50])
# features = freq_band_features(dataset, freq_bands)
# features = hjorth_features(dataset)
# features = entropy_features(dataset)
import mne_features.univariate as mne_f

def fractal_features(data):
    '''
    Computes the Higuchi Fractal Dimension and Katz Fractal Dimension using the package mne_features.

    Args:
        data (ndarray): EEG data.

    Returns:
        ndarray: Computed features.

    '''
    n_trials, n_secs, n_channels, _ = data.shape
    print(n_trials, n_secs, n_channels)
    features_per_channel = 2

    features = np.empty([n_trials, n_secs, n_channels * features_per_channel])
    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            higuchi = mne_f.compute_higuchi_fd(second)
            katz = mne_f.compute_katz_fd(second)
            features[i][j] = np.concatenate([higuchi, katz])
    features = features.reshape(
        [n_trials*n_secs, n_channels*features_per_channel])
    return features

def freq_band_features(data, freq_bands):
    '''
    Computes the frequency bands delta, theta, alpha, beta and gamma using the package mne_features.

    Args:
        data (ndarray): EEG data.
        freq_bands (ndarray): The frequency bands to compute.

    Returns:
        ndarray: Computed features.
    '''
    n_trials, n_secs, n_channels, sfreq = data.shape
    features_per_channel = len(freq_bands)-1

    features = np.empty([n_trials, n_secs, n_channels * features_per_channel])
    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            psd = mne_f.compute_pow_freq_bands(
                sfreq, second, freq_bands=freq_bands)
            features[i][j] = psd
    features = features.reshape(
        [n_trials*n_secs, n_channels*features_per_channel])
    return features

features = freq_band_features(dataset)
n_epochs = features.shape[0] // labels.shape[0]   # = 25

labels_expanded = np.repeat(labels, n_epochs)
print(len(labels_expanded))


NameError: name 'dataset' is not defined

In [174]:
data = features

# k-NN Classifier

In [ ]:
x, x_test, y, y_test = train_test_split(
    data, labels_expanded, test_size=0.2, random_state=1)
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.25, random_state=1)
scaler = MinMaxScaler()
scaler.fit(x_train)
x = scaler.transform(x)
x_train = scaler.transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

param_grid = {
    'leaf_size': range(50),
    'n_neighbors': range(1, 10),
    'p': [1, 2]
}
split_index = [-1 if x in range(len(x_train)) else 0 for x in range(len(x))]
ps = PredefinedSplit(test_fold=split_index)
knn_clf = GridSearchCV(KNeighborsClassifier(), param_grid, cv=ps, refit=True)
knn_clf.fit(x, y)

In [ ]:
y_pred = knn_clf.predict(x_test)
y_true = y_test

In [ ]:
print(metrics.classification_report(y_true, y_pred))
print(metrics.confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

       False       0.57      0.55      0.56       311
        True       0.54      0.56      0.55       289

    accuracy                           0.56       600
   macro avg       0.56      0.56      0.55       600
weighted avg       0.56      0.56      0.56       600

[[170 141]
 [126 163]]


# SVM Classifier

In [ ]:
x, x_test, y, y_test = train_test_split(
    data, labels_expanded, test_size=0.2, random_state=1)
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.25, random_state=1)

param_grid = {
    'C': [0.1, 1, 10, 100, 1000],
    'kernel': ['rbf']
}
split_index = [-1 if x in range(len(x_train)) else 0 for x in range(len(x))]
ps = PredefinedSplit(test_fold=split_index)
svm_clf = GridSearchCV(SVC(), param_grid, cv=ps, refit=True)
svm_clf.fit(x, y)




,estimator,SVC()
,param_grid,"{'C': [0.1, 1, ...], 'kernel': ['rbf']}"
,scoring,None
,n_jobs,None
,refit,True
,cv,"PredefinedSpl...hape=(9600,)))"
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,10


In [182]:
import joblib
joblib.dump(svm_clf, "svm_model.pkl")
y_pred = svm_clf.predict(x_test)
y_true = y_test

In [181]:
print(metrics.classification_report(y_true, y_pred))
print(metrics.confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.75      0.98      0.85      1777
           1       0.62      0.08      0.14       623

    accuracy                           0.75      2400
   macro avg       0.69      0.53      0.49      2400
weighted avg       0.72      0.75      0.67      2400

[[1748   29]
 [ 575   48]]


# Multilayer Perceptron

In [ ]:
K.clear_session()
y_v = label
y_v = to_categorical(y_v)
x_train, x_test, y_train, y_test = train_test_split(
    data, y_v, test_size=0.2, random_state=1)
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.25, random_state=1)

In [ ]:
def model_builder(hp):
    model = models.Sequential()
    model.add(Input(shape=(x_train.shape[1],)))

    for i in range(hp.Int('layers', 2, 6)):
        model.add(Dense(units=hp.Int('units_' + str(i), 32, 1024, step=32),
                        activation=hp.Choice('act_' + str(i), ['relu', 'sigmoid'])))

    model.add(Dense(v.N_CLASSES, activation='softmax', name='out'))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(optimizer=opt.adam_v2.Adam(learning_rate=hp_learning_rate),
                  loss="binary_crossentropy",
                  metrics=['accuracy'])
    return model

In [ ]:
tuner = RandomSearch(
    model_builder,
    objective='val_accuracy',
    max_trials=15,
    executions_per_trial=2,
    overwrite=True
)

In [ ]:
tuner.search(x_train, y_train, epochs=50, validation_data=[x_val, y_val])

Trial 15 Complete [00h 01m 08s]
val_accuracy: 0.5450000166893005

Best val_accuracy So Far: 0.5541666746139526
Total elapsed time: 00h 14m 40s
INFO:tensorflow:Oracle triggered exit


In [ ]:
model = tuner.get_best_models(num_models=1)[0]

In [ ]:
y_pred = model.predict(x_test)
y_true = y_test
y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_true, axis=1)

2022-12-11 13:24:08.707790: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:113] Plugin optimizer for device_type GPU is enabled.


In [ ]:
print(metrics.classification_report(y_true, y_pred))
print(metrics.confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.52      0.90      0.66       311
           1       0.52      0.11      0.19       289

    accuracy                           0.52       600
   macro avg       0.52      0.51      0.43       600
weighted avg       0.52      0.52      0.43       600

[[281  30]
 [256  33]]
